In [44]:
import os, subprocess
import rasterio
import rioxarray as rxr
import xarray as xr
import geopandas as gpd
import numpy as np
import zarr
import dask
from dask.distributed import Client
from pathlib import Path
from rasterio.windows import from_bounds
from xrspatial import slope, aspect

In [33]:
#Path to the COG #Chemin vers le fichier COG
cog_path = 'https://canelevation-dem.s3.ca-central-1.amazonaws.com/mrdem-30/mrdem-30-dtm.tif'

In [37]:
# Get AOI bounds from ecozones — reprojected to match the MrDEM COG CRS (EPSG:3979).
# The COG uses Canada Lambert Conformal Conic (coordinates in metres).
# The geojson is WGS84 (degrees). Passing WGS84 lon/lat values to from_bounds()
# with a metre-based transform produces windows outside the raster extent → empty.
ecozones = gpd.read_file("../data/Ecozones_of_Canada.geojson")

MR_DEM_CRS = "EPSG:3979"   # Canada Lambert Conformal Conic — matches the COG
id_field   = "ZONE_NAME"

# Reproject all ecozone geometries to the raster CRS before extracting bounds
ecozones_proj = ecozones.to_crs(MR_DEM_CRS)

aoi_regions = {
    row[id_field]: row.geometry.bounds      # bounds are now in metres (EPSG:3979)
    for _, row in ecozones_proj.iterrows()
    if row.geometry and not row.geometry.is_empty
}

aoi_regions["Hudson Plain"]

(-46552.25574289911,
 176496.91580375933,
 1272065.0430560056,
 1110179.3407448374)

In [21]:
curr_dir = Path.cwd()
output_dir = Path(curr_dir.parent.parent.parent / "data/ecozone_dems")
print(output_dir)
os.makedirs(output_dir, exist_ok=True)

/Users/jgoldman/Documents/data/ecozone_dems


In [38]:


with rasterio.open(cog_path) as src:

    for name, (minx, miny, maxx, maxy) in aoi_regions.items():

        print(f"Processing {name}")

        # Create raster window from AOI bounds
        window = from_bounds(minx, miny, maxx, maxy, src.transform)

        # Read data
        data = src.read(window=window)

        if data.size == 0:
            print(f"  Skipping {name} (empty window)")
            continue

        # Update metadata
        transform = src.window_transform(window)
        meta = src.meta.copy()
        meta.update({
            "height": data.shape[1],
            "width": data.shape[2],
            "transform": transform
        })

        # Write output
        out_tif = output_dir / f"{name}.tif"
        with rasterio.open(out_tif, "w", **meta) as dst:
            dst.write(data)



Processing Northern Arctic
Processing Arctic Cordillera
Processing Southern Arctic
Processing Taiga Cordillera
Processing Taiga Plain
Processing Taiga Shield
Processing Boreal Cordillera
Processing Boreal PLain
Processing Pacific Maritime
Processing Boreal Shield
Processing Hudson Plain
Processing Montane Cordillera
Processing Prairie
Processing Atlantic Maritime
Processing MixedWood Plain


In [39]:
aoi_regions["Boreal PLain"]

(-1364683.3349427718,
 376984.6595887976,
 -1330248.4606606779,
 448633.09694683587)

## Terrain Derivatives → Zarr

For each downloaded ecozone DEM, we compute **elevation, slope, and aspect**
and write all three as variables into a single `.zarr` store with Zstd compression.

**Pipeline:**
- `rioxarray` opens each TIF lazily with dask chunks (no full file load)
- `xrspatial.slope` / `xrspatial.aspect` use Horn (1981) with `dask.array.map_overlap`
  so chunk boundaries are handled correctly (1-pixel ghost zone)
- Output uses `zarr.codecs.ZstdCodec(level=3)` — good speed/compression trade-off

**CRS note:** slope and aspect are only correct when raster units are metres.
MrDEM is EPSG:3979 (Lambert Conformal Conic, metres) — correct as downloaded.

In [45]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Input:  directory containing the per-ecozone TIFs downloaded above
# Output: same directory, one .zarr store per ecozone

CHUNK_SIZE = 4096   # pixels per chunk side — tune to your RAM
                    # 4096 × 4096 × float32 × 3 vars ≈ 192 MB per chunk

ZSTD_LEVEL = 3      # 1 = fastest, 22 = max compression; 3 is a good default

# Reuse output_dir defined in the download section
zarr_dir = output_dir.parent / "ecozone_zarr"
zarr_dir.mkdir(parents=True, exist_ok=True)
print(f"Input TIFs : {output_dir}")
print(f"Output Zarr: {zarr_dir}")

tif_files = sorted(output_dir.glob("*.tif"))
print(f"Found {len(tif_files)} TIF file(s): {[f.name for f in tif_files]}")

Input TIFs : /Users/jgoldman/Documents/data/ecozone_dems
Output Zarr: /Users/jgoldman/Documents/data/ecozone_zarr
Found 15 TIF file(s): ['Arctic Cordillera.tif', 'Atlantic Maritime.tif', 'Boreal Cordillera.tif', 'Boreal PLain.tif', 'Boreal Shield.tif', 'Hudson Plain.tif', 'MixedWood Plain.tif', 'Montane Cordillera.tif', 'Northern Arctic.tif', 'Pacific Maritime.tif', 'Prairie.tif', 'Southern Arctic.tif', 'Taiga Cordillera.tif', 'Taiga Plain.tif', 'Taiga Shield.tif']


In [46]:
def tif_to_terrain_zarr(
    tif_path: Path,
    zarr_path: Path,
    chunk_size: int = 4096,
    zstd_level: int = 3,
) -> None:
    """
    Compute elevation, slope, and aspect from a DEM TIF and write to Zarr.

    Parameters
    ----------
    tif_path   : input DEM GeoTIFF (units must be metres for correct slope/aspect)
    zarr_path  : output .zarr store path (overwritten if it exists)
    chunk_size : spatial chunk size in pixels (square chunks)
    zstd_level : Zstd compression level (1–22)
    """
    print(f"\n{'─'*60}")
    print(f"Input : {tif_path.name}")
    print(f"Output: {zarr_path.name}")

    # ── Open DEM lazily — no data loaded yet ──────────────────────────────
    dem = (
        rxr.open_rasterio(
            tif_path,
            chunks={"x": chunk_size, "y": chunk_size},
            lock=False,       # allow concurrent chunk reads
        )
        .squeeze("band", drop=True)   # single-band DEM → (y, x)
        .astype("float32")
    )

    # Guard: slope/aspect require metre units
    crs = dem.rio.crs
    if crs and crs.is_geographic:
        raise ValueError(
            f"{tif_path.name} is in a geographic CRS ({crs}). "
            "Reproject to a projected CRS in metres before computing slope/aspect."
        )

    print(f"  CRS   : {crs}")
    print(f"  Shape : {dem.sizes}")
    print(f"  Chunks: {dict(dem.chunksizes)}")

    # ── Terrain derivatives ────────────────────────────────────────────────
    # xrspatial uses dask.array.map_overlap internally — chunk edges handled
    slope_da  = slope(dem).astype("float32")   # degrees 0–90
    aspect_da = aspect(dem).astype("float32")  # degrees 0–360, clockwise from N

    # ── Assemble dataset ───────────────────────────────────────────────────
    ds = xr.Dataset({
        "elevation": dem,
        "slope":     slope_da,
        "aspect":    aspect_da,
    })
    # Carry spatial metadata forward
    ds = ds.rio.write_crs(crs)

    # ── Zarr encoding with Zstd ────────────────────────────────────────────
    compressor = zarr.codecs.ZstdCodec(level=zstd_level)
    encoding   = {
        var: {
            "compressors": compressor,
            "chunks":      [chunk_size, chunk_size],
        }
        for var in ds.data_vars
    }

    # ── Write (triggers dask computation) ─────────────────────────────────
    ds.to_zarr(zarr_path, mode="w", encoding=encoding)

    size_mb = sum(f.stat().st_size for f in zarr_path.rglob("*") if f.is_file()) / 1e6
    print(f"  Done  : {zarr_path.name}  ({size_mb:.0f} MB on disk)")

In [58]:
# ── Start a local Dask cluster for parallel chunk processing ──────────────────
# Adjust n_workers and memory_limit to your machine.
# Remove / comment out if you prefer single-threaded (uses less RAM).
client = Client(n_workers=4, threads_per_worker=2, memory_limit="8GB")
print(client)
print(f"Dashboard: {client.dashboard_link}")

<Client: 'tcp://127.0.0.1:58753' processes=4 threads=8, memory=29.80 GiB>
Dashboard: http://127.0.0.1:8787/status


2026-04-16 10:00:48,661 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:58765 (pid=34123) exceeded 95% memory budget. Restarting...
2026-04-16 10:00:48,844 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:58765' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 11, 0.09999999999999998), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 3.9, 11.1), ('store-map-97ea1a396eeb9b6b7a7fad36bf4cf082', 7, 14), ('store-map-4a9f166a5db7ad3d894a212b02b5fea4', 5, 30), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 11.1, 0.09999999999999998), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 3.9, 2.1), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 11.1, 22.9), ('store-map-4a9f166a5db7ad3d894a212b02b5fea4', 6, 21), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 4, 21), ('getitem-0bd56214fc0f4e12a5e41fb3aa73ba06', 4, 18.9), ('store-map-fc81752f7ea74ce03518f1f3763aa5c0', 6, 19), ('store-map-4a9f166a5

In [67]:
# - process single ecozone
zone_name = "Southern Arctic"

tif_path = output_dir/ f"{zone_name}.tif"
zarr_path = zarr_dir / f"{zone_name}.zarr"

tif_to_terrain_zarr(
        tif_path   = tif_path,
        zarr_path  = zarr_path,
        chunk_size = CHUNK_SIZE,
        zstd_level = ZSTD_LEVEL,
    )




────────────────────────────────────────────────────────────
Input : Southern Arctic.tif
Output: Southern Arctic.zarr
  CRS   : EPSG:3979
  Shape : Frozen({'y': 73772, 'x': 100914})
  Chunks: {'y': (4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 44), 'x': (4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 4096, 2610)}


/Users/jgoldman/Documents/postdoc/peatland_fire_selectivity/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-04-16 10:59:49,750 - distributed.worker.memory - WARNING - Worker is at 84% memory usage. Pausing worker.  Process memory: 6.28 GiB -- Worker memory limit: 7.45 GiB
2026-04-16 10:59:50,573 - distributed.worker.memory - WARNING - Worker is at 77% memory usage. Resuming worker. Process memory: 5.76 GiB -- Worker memory limit: 7.45 GiB
2026-04-16 10:59:59,159 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 6.01 GiB -- Worker memory limit: 7.45 GiB
2026-04-16 11:00:00,480 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 6.00 GiB -- Worker memory limit: 7.45 G

  Done  : Southern Arctic.zarr  (73619 MB on disk)


In [ ]:
# ── Process all ecozone TIFs ──────────────────────────────────────────────────
# Skips any zone whose .zarr already exists — safe to re-run after interruption.

for tif_path in tif_files:
    zarr_path = zarr_dir / (tif_path.stem + ".zarr")

    if zarr_path.exists():
        print(f"Skipping {tif_path.name} (zarr already exists)")
        continue

    tif_to_terrain_zarr(
        tif_path   = tif_path,
        zarr_path  = zarr_path,
        chunk_size = CHUNK_SIZE,
        zstd_level = ZSTD_LEVEL,
    )

print("\nAll zones complete.")
client.close()

In [ ]:
# ── Quick verification — open one zarr and inspect ────────────────────────────
sample_zarr = next(zarr_dir.glob("*.zarr"), None)

if sample_zarr:
    ds_check = xr.open_zarr(sample_zarr)
    print(ds_check)
    print()
    for var in ds_check.data_vars:
        arr = ds_check[var]
        print(f"{var:12s}  shape={arr.shape}  dtype={arr.dtype}  "
              f"min={float(arr.min()):8.2f}  max={float(arr.max()):8.2f}")
else:
    print("No zarr stores found yet.")

make sure everything is closed

In [57]:
from dask.distributed import client as dask_client_module

for c in list(dask_client_module._global_clients.values()):
      c.close()

2026-04-16 09:59:50,593 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:58716'. Reason: nanny-close
2026-04-16 09:59:50,593 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-16 09:59:50,593 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:58718'. Reason: nanny-close
2026-04-16 09:59:50,594 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-16 09:59:50,594 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:58720'. Reason: nanny-close
2026-04-16 09:59:50,595 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-16 09:59:50,595 - distributed.nanny - INFO - Closing Nanny at 'tcp://127.0.0.1:58722'. Reason: nanny-close
2026-04-16 09:59:50,595 - distributed.nanny - INFO - Nanny asking worker to close. Reason: nanny-close
2026-04-16 09:59:50,643 - distributed.nanny - INFO - Nanny at 'tcp://127.0.0.1:58720' closed.
2026-04-16 09:59:50,646 - distribu